In [ ]:
# 0. Extract Notebook 01 outputs (if a ZIP was uploaded)
from pathlib import Path
import zipfile

zip_path = Path('statistical_outputs.zip')
required_relative = Path('prepared_company_quarters.csv')
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as archive:
        invalid = [name for name in archive.namelist() if '..' in Path(name).parts or Path(name).is_absolute()]
        if invalid:
            raise ValueError('Archive contains unsafe paths: ' + repr(invalid[:3]))
        has_root = any(Path(name).parts[:1] == ('statistical_outputs',) for name in archive.namelist())
        dest = Path('.') if has_root else Path('statistical_outputs')
        archive.extractall(dest)
        print(f'Extracted {zip_path} into {dest}')
else:
    print('No statistical_outputs.zip found. Using existing statistical_outputs directory.')
assert (Path('statistical_outputs') / required_relative).exists() or (
    Path('/content/statistical_outputs') / required_relative).exists(),     'Notebook 01 output not found after extraction.'


# 02 — Model Training and Validation (FINAL WORKFLOW — PRE-TEST VALIDATION)

This notebook deliberately separates **data splitting**, **feature preparation**, **individual model experiments**, and **final comparison**.

**Run order**
1. Load Notebook 01's prepared company-quarter data and create distinct train, validation and reserved test datasets.
2. Select candidate features using **training-only availability**.
3. Train each model independently, with its own code cell and saved results.
4. Compare all validation results and freeze the selected model configurations.
5. Optionally evaluate banking-sector variables separately.

**Research integrity:** Only training data are used to fit candidate models. Validation results select hyperparameters and model family. The reserved test partition is neither fitted nor scored here; Notebook 03 evaluates it once after decisions are frozen.


**New in this review:** Exact quarter-over-quarter delta and growth-rate features for every eligible reported financial metric, including banking-specific variables in the sector experiment. The canonical `reported_only` set is unchanged; `reported_plus_engineered` is expanded.
**Added experiment:** exact same-quarter-last-year (lag-4) values, year-over-year changes and growth, and target-quarter sine/cosine encodings. The seasonal feature set is tested separately to isolate its contribution.


## 1. Load data and explicitly create train / validation / test datasets

The input data are enriched with exact-quarter growth fields **before splitting**, using only historical/current-quarter feature values. Splits are still based on the original target quarter. No future PAT target information is used.

In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

SEED=640
SOURCE=Path('statistical_outputs')
if not (SOURCE/'prepared_company_quarters.csv').exists(): SOURCE=Path('/content/statistical_outputs')
assert (SOURCE/'prepared_company_quarters.csv').exists(), 'Run Notebook 01 and place statistical_outputs beside this notebook.'
prepared=SOURCE/'prepared_company_quarters.csv'
handoff=json.loads((SOURCE/'handoff_manifest.json').read_text())
source_hash=hashlib.sha256(prepared.read_bytes()).hexdigest()
assert source_hash==handoff['prepared_csv_sha256'], 'Input file checksum mismatch'
df=pd.read_csv(prepared,low_memory=False)
# Generate *as-of-quarter* changes using the EXACT preceding calendar quarter.
# This function uses source financial values and identifiers only; NEVER PAT_next/target_growth_pct.
QOQ_CORE = ['PAT','REVENUE','TOTAL_INCOME','OTHER_INCOME','PROFIT_BEFORE_TAX',
            'TAX_EXPENSE','EXPENSES','FINANCE_COSTS','EPS_BASIC','EPS_DILUTED',
            'ASSETS','LIABILITIES','EQUITY','BORROWINGS']
QOQ_BANK = ['OPERATING_PROFIT_BEFORE_PROVISIONS','INTEREST_EARNED',
            'INTEREST_EXPENDED','PROVISIONS','GROSS_NPA','NET_NPA']
QOQ_SOURCE_COLUMNS = QOQ_CORE + QOQ_BANK
QOQ_SUFFIXES = ['_qoq_abs_change', '_qoq_growth_pct']

def add_exact_quarter_growth(panel):
    required = {'COMPANY_KEY','q'}
    assert required.issubset(panel.columns), 'Prepared dataset must have COMPANY_KEY and q.'
    out = panel.copy()
    out['_q_period'] = pd.PeriodIndex(out['q'].astype(str),freq='Q')
    assert not out.duplicated(['COMPANY_KEY','_q_period']).any(), \
        'Ambiguous company-quarter rows: fix source before computing QoQ.'
    present = [x for x in QOQ_SOURCE_COLUMNS if x in out.columns]
    for x in present:
        out[x] = pd.to_numeric(out[x], errors='coerce').replace([np.inf,-np.inf],np.nan)
    # Drop any pre-existing generated fields, to guarantee deterministic recomputation.
    for x in present:
        out.drop(columns=[x+s for s in QOQ_SUFFIXES],inplace=True,errors='ignore')
    # Shift the previous quarter's key FORWARD by one quarter, then join to current.
    # This prevents false QoQ growth across gaps such as Q1 -> Q3.
    prev = out[['COMPANY_KEY','_q_period']+present].copy()
    prev['_q_period'] = prev['_q_period']+1
    prev.rename(columns={x:'_previous_'+x for x in present},inplace=True)
    out=out.merge(prev,on=['COMPANY_KEY','_q_period'],how='left',validate='one_to_one',sort=False)
    for x in present:
        previous=out.pop('_previous_'+x)
        current=out[x]
        out[x+'_qoq_abs_change']=current-previous
        # For losses and negative base values, use abs(previous), matching the study's target convention.
        denominator=previous.abs().where(previous.abs()>0)
        out[x+'_qoq_growth_pct']=100*(current-previous)/denominator
    out.drop(columns=['_q_period'],inplace=True)
    new_fields=[x+s for x in present for s in QOQ_SUFFIXES]
    out[new_fields]=out[new_fields].replace([np.inf,-np.inf],np.nan)
    return out, new_fields

# Same-calendar-quarter-last-year features; never use future quarter values.
SEASONAL_SOURCE_COLUMNS = QOQ_SOURCE_COLUMNS
SEASONAL_SUFFIXES = ['_lag4', '_yoy_abs_change', '_yoy_growth_pct']
SEASONAL_CALENDAR = ['target_quarter_sin', 'target_quarter_cos']

def add_exact_annual_seasonality(panel):
    out = panel.copy()
    out['_season_q'] = pd.PeriodIndex(out['q'].astype(str),freq='Q')
    assert not out.duplicated(['COMPANY_KEY','_season_q']).any(), 'Duplicate company-quarter values'
    present = [c for c in SEASONAL_SOURCE_COLUMNS if c in out.columns]
    for c in present:
        out[c] = pd.to_numeric(out[c], errors='coerce').replace([np.inf,-np.inf], np.nan)
    drop_cols = [c+s for c in present for s in SEASONAL_SUFFIXES] + SEASONAL_CALENDAR
    out.drop(columns=drop_cols, errors='ignore', inplace=True)
    lag = out[['COMPANY_KEY','_season_q']+present].copy()
    lag['_season_q'] = lag['_season_q'] + 4
    lag.rename(columns={c:'_lag4_source_'+c for c in present}, inplace=True)
    out = out.merge(lag, on=['COMPANY_KEY','_season_q'],how='left', validate='one_to_one',sort=False)
    for c in present:
        prev = out.pop('_lag4_source_'+c)
        cur = out[c]
        out[c+'_lag4'] = prev
        out[c+'_yoy_abs_change'] = cur-prev
        out[c+'_yoy_growth_pct'] = 100*(cur-prev)/prev.abs().where(prev.abs()>0)
    # The date whose PAT growth is being predicted is q+1, never read its value.
    tq = (out['_season_q']+1).dt.quarter.astype(float)
    out['target_quarter_sin'] = np.sin(2*np.pi*tq/4)
    out['target_quarter_cos'] = np.cos(2*np.pi*tq/4)
    out.drop(columns=['_season_q'],inplace=True)
    generated = [c+s for c in present for s in SEASONAL_SUFFIXES] + SEASONAL_CALENDAR
    out[generated] = out[generated].replace([np.inf,-np.inf],np.nan)
    return out, generated

df, QOQ_GENERATED = add_exact_quarter_growth(df)
df, SEASONAL_GENERATED = add_exact_annual_seasonality(df)
assert {'split','COMPANY_KEY','target_q','target_growth_pct','INDEX_INDUSTRY','quarter_of_year'}<=set(df)
df=df[df.split.isin(['train','validation','test'])].copy()
assert df.target_growth_pct.notna().all()
q={s:set(df.loc[df.split.eq(s),'target_q'].astype(str)) for s in ('train','validation','test')}
assert all(q.values()) and max(q['train'])<min(q['validation'])<min(q['test'])
assert not any(q[a]&q[b] for a,b in [('train','validation'),('train','test'),('validation','test')])
OUT=SOURCE.parent/'model_outputs';OUT.mkdir(exist_ok=True,parents=True)
TRAIN_DATA = df.loc[df.split.eq('train')].copy().reset_index(drop=True)
VALIDATION_DATA = df.loc[df.split.eq('validation')].copy().reset_index(drop=True)
TEST_DATA = df.loc[df.split.eq('test')].copy().reset_index(drop=True)

assert len(TRAIN_DATA) and len(VALIDATION_DATA) and len(TEST_DATA)
assert len(TRAIN_DATA)+len(VALIDATION_DATA)+len(TEST_DATA)==len(df)
assert set(TRAIN_DATA.target_q).isdisjoint(set(VALIDATION_DATA.target_q))
assert set(TEST_DATA.target_q).isdisjoint(set(TRAIN_DATA.target_q) | set(VALIDATION_DATA.target_q))

SPLITS = OUT / 'data_splits'
SPLITS.mkdir(exist_ok=True)
TRAIN_DATA.to_csv(SPLITS/'train.csv',index=False)
VALIDATION_DATA.to_csv(SPLITS/'validation.csv',index=False)
TEST_DATA.to_csv(SPLITS/'test_reserved_do_not_tune.csv',index=False)

split_summary=pd.DataFrame([
    {'split':'train', 'rows':len(TRAIN_DATA), 'companies':TRAIN_DATA.COMPANY_KEY.nunique(),
     'first_target_q':min(q['train']),'last_target_q':max(q['train'])},
    {'split':'validation', 'rows':len(VALIDATION_DATA), 'companies':VALIDATION_DATA.COMPANY_KEY.nunique(),
     'first_target_q':min(q['validation']),'last_target_q':max(q['validation'])},
    {'split':'test (reserved)', 'rows':len(TEST_DATA), 'companies':TEST_DATA.COMPANY_KEY.nunique(),
     'first_target_q':min(q['test']),'last_target_q':max(q['test'])},
])
split_summary.to_csv(OUT/'split_summary.csv',index=False)
display(split_summary)
print('Test partition is saved for Notebook 03 but not used for model training or model selection.')

## 2. Select features and define shared training utilities

The enhanced model additionally includes training-eligible **absolute quarter-over-quarter changes** and **percentage quarter-over-quarter growth** for every available core financial item. Banking-only QoQ features are considered separately in the Financial Services experiment. Percentage growth is missing when the immediately preceding quarter is absent or its denominator is zero; the training-fold median imputer handles this without examining validation/test targets.

Seasonal candidate features are eligible only from training observations, not from validation or test statistics.


In [ ]:
from sklearn.linear_model import HuberRegressor
CORE=['PAT','REVENUE','TOTAL_INCOME','OTHER_INCOME','PROFIT_BEFORE_TAX','TAX_EXPENSE',
      'EXPENSES','FINANCE_COSTS','EPS_BASIC','EPS_DILUTED','ASSETS','LIABILITIES','EQUITY','BORROWINGS']
BANK=['OPERATING_PROFIT_BEFORE_PROVISIONS','INTEREST_EARNED','INTEREST_EXPENDED',
      'PROVISIONS','GROSS_NPA','NET_NPA']
ENGINEERED=['pat_growth_pct','revenue_growth_pct','eps_basic_growth_pct',
            'pat_margin_pct','finance_cost_income_pct','expense_income_pct','negative_base_pat']
QOQ_CORE_ENGINEERED = [c for c in QOQ_GENERATED if c.split('_qoq_')[0] in CORE]
QOQ_BANK_ENGINEERED = [c for c in QOQ_GENERATED if c.split('_qoq_')[0] in BANK]
ENGINEERED = list(dict.fromkeys(ENGINEERED + QOQ_CORE_ENGINEERED))
SEASONAL_CORE_ENGINEERED = [c for c in SEASONAL_GENERATED if c.endswith(('_lag4','_yoy_abs_change','_yoy_growth_pct')) and c.rsplit('_',1)[0] is not None and any(c.startswith(x+'_') for x in CORE)]
SEASONAL_BANK_ENGINEERED = [c for c in SEASONAL_GENERATED if any(c.startswith(x+'_') for x in BANK)]
SEASONAL_CORE_ENGINEERED += SEASONAL_CALENDAR
CATS=['INDEX_INDUSTRY','quarter_of_year']
audit_path=SOURCE/'expanded_statistical_analysis'/'feature_eligibility_inventory.csv'
assert audit_path.exists(), 'Rerun updated Notebook 01 to export its expanded feature audit.'
external_audit=pd.read_csv(audit_path)
assert {'feature','status','role'}<=set(external_audit)
for c in CORE+BANK+ENGINEERED+QOQ_BANK_ENGINEERED+SEASONAL_GENERATED:
 if c in df: df[c]=pd.to_numeric(df[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
for c in CATS: df[c]=df[c].fillna('Unknown').astype(str)
train=df.loc[df.split.eq('train')].copy()
valid=df.loc[df.split.eq('validation')].copy()
# TEST_DATA is deliberately not transformed, fitted or scored in Notebook 02.
def eligibility(data,cols,min_rows=80,min_firms=20):
 eligible=[];rows=[]
 for c in cols:
  if c not in data: rows.append({'feature':c,'included':False,'reason':'not present'});continue
  x=data[c].dropna();companies=data.loc[x.index,'COMPANY_KEY'].nunique()
  ok=len(x)>=min_rows and companies>=min_firms and x.nunique()>=3
  rows.append({'feature':c,'included':bool(ok),'nonmissing_train':len(x),'n_companies':int(companies),
               'distinct':x.nunique(),'reason':'eligible' if ok else 'too sparse or constant'})
  if ok:eligible.append(c)
 return eligible,pd.DataFrame(rows)
reported,core_audit=eligibility(train,CORE)
engineered,eng_audit=eligibility(train,ENGINEERED)
assert reported and engineered,'Insufficient valid reported or engineered columns for RQ4.'
seasonal, seasonal_audit = eligibility(train,SEASONAL_CORE_ENGINEERED)
assert seasonal, 'No seasonal features pass training-only eligibility; inspect quarter coverage.'
FEATURE_SETS={'reported_only':reported+CATS,
 'reported_plus_engineered':reported+engineered+CATS,
 'reported_plus_engineered_plus_seasonal':reported+engineered+seasonal+CATS}
pd.concat([core_audit,eng_audit,seasonal_audit]).to_csv(OUT/'training_feature_eligibility.csv',index=False)
qoq_summary = pd.DataFrame([{'variable':c,'nonmissing_train':int(train[c].notna().sum()),
    'training_missing_pct':100*train[c].isna().mean(),
    'included_in_enhanced_model':c in engineered} for c in QOQ_CORE_ENGINEERED])
qoq_summary.to_csv(OUT/'qoq_feature_eligibility.csv',index=False)
display(qoq_summary.head(25))
display(pd.concat([core_audit,eng_audit,seasonal_audit])); print(FEATURE_SETS)
seasonal_audit.to_csv(OUT/'seasonality_feature_eligibility.csv',index=False)
print('Seasonal feature count:',len(seasonal))

def preprocessor(features):
 nums=[c for c in features if c not in CATS];cats=[c for c in features if c in CATS]
 try: enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
 except TypeError:enc=OneHotEncoder(handle_unknown='ignore',sparse=False)
 return ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),
         ('scale',StandardScaler())]),nums),('cat',enc,cats)])
def pipeline(est,cols):return Pipeline([('prep',preprocessor(cols)),('model',est)])
def estimator(name,p):
 if name=='Historical median':return DummyRegressor(**p)
 if name=='Linear Regression':return LinearRegression()
 if name=='Ridge':return Ridge(**p)
 if name=='Random Forest':return RandomForestRegressor(random_state=SEED,n_jobs=-1,**p)
 if name=='Gradient Boosting':return GradientBoostingRegressor(random_state=SEED,**p)
 if name=='GB absolute error':return GradientBoostingRegressor(loss='absolute_error',random_state=SEED,**p)
 if name=='Huber':return HuberRegressor(**p)
 raise ValueError(name)
def metrics(y,p):return dict(MAE=float(mean_absolute_error(y,p)),
 RMSE=float(np.sqrt(mean_squared_error(y,p))),R2=float(r2_score(y,p)))

## 3. Independent model experiments

Each model gets its **own cell**, using the same frozen train and validation partitions and the same eligibility-selected feature sets. Within each model, hyperparameter configurations are evaluated against validation data. Detailed configuration-level results and best-per-feature-set predictions are saved separately.

The `MODEL_KEY` is a stable readable name such as `ridge` or `random_forest`. This is different from a model configuration's unique key (`feature_set|algorithm|parameters`), which is kept for Notebook 03 compatibility.

**Run the cells in order** so the final comparison has all five sets of results. Re-running one model cell replaces that model's saved outputs rather than duplicating records.


In [ ]:
GRID = {
    'Historical median':[{'strategy':'median'}],
    'Linear Regression':[{}],
    'Ridge':list(ParameterGrid({'alpha':[1.,10.,100.]})),
    'Random Forest':list(ParameterGrid({
        'n_estimators':[150], 'max_depth':[5,10], 'min_samples_leaf':[5,15]})),
    'Gradient Boosting':list(ParameterGrid({
        'n_estimators':[100,200], 'learning_rate':[.03,.08],
        'max_depth':[2], 'min_samples_leaf':[10]})),
    # Primary additions motivated by the development-only robust pilot. No test outcomes used.
    'GB absolute error':list(ParameterGrid({
        'n_estimators':[100,150,250], 'learning_rate':[.03,.05],
        'max_depth':[2], 'min_samples_leaf':[10,15]})),
    'Huber':list(ParameterGrid({'epsilon':[1.35,1.75], 'alpha':[1.0,10.0], 'max_iter':[2000]}))
}
EXPERIMENTS_DIR=OUT/'model_experiments'
EXPERIMENTS_DIR.mkdir(exist_ok=True)
model_results={}

def run_model_experiment(model_key, algorithm):
    """Fit/tune one algorithm only; save its own scores, validation predictions and pipeline."""
    assert algorithm in GRID
    records=[]
    val_prediction_rows=[]
    fitted={}
    for feature_set,cols in FEATURE_SETS.items():
        for params in GRID[algorithm]:
            model=pipeline(estimator(algorithm,params),cols)
            model.fit(train[cols],train.target_growth_pct)
            predictions=model.predict(valid[cols])
            score=metrics(valid.target_growth_pct,predictions)
            key=f'{feature_set}|{algorithm}|{json.dumps(params,sort_keys=True)}'
            row={'key':key,'model_key':model_key,'feature_set':feature_set,
                 'algorithm':algorithm,'params_json':json.dumps(params,sort_keys=True),**score}
            records.append(row)
            fitted[key]=model
            val_prediction_rows.append(pd.DataFrame({
                'key':key,
                'COMPANY_KEY':valid.COMPANY_KEY.to_numpy(),
                'target_q':valid.target_q.astype(str).to_numpy(),
                'actual':valid.target_growth_pct.to_numpy(),
                'predicted':predictions
            }))
    trials_one=pd.DataFrame(records).sort_values(
        ['MAE','RMSE','key']).reset_index(drop=True)
    assert not trials_one.empty
    assert np.isfinite(trials_one[['MAE','RMSE','R2']].to_numpy()).all()
    preds_one=pd.concat(val_prediction_rows,ignore_index=True)
    best_one=trials_one.drop_duplicates('feature_set').copy()
    model_dir=EXPERIMENTS_DIR/model_key
    model_dir.mkdir(parents=True,exist_ok=True)
    trials_one.to_csv(model_dir/'validation_trials.csv',index=False)
    preds_one.to_csv(model_dir/'validation_predictions.csv',index=False)
    best_one.to_csv(model_dir/'best_by_feature_set.csv',index=False)
    for _,row in best_one.iterrows():
        joblib.dump(fitted[row['key']],
                    model_dir/('best_'+row['feature_set']+'.joblib'))
    (model_dir/'experiment_manifest.json').write_text(json.dumps({
        'model_key':model_key,'algorithm':algorithm,
        'input_sha256':source_hash,
        'train_target_quarters':sorted(q['train']),
        'validation_target_quarters':sorted(q['validation']),
        'test_not_used':True,
        'feature_sets':FEATURE_SETS,
        'best_keys_by_feature_set':dict(zip(best_one.feature_set,best_one.key))
    },indent=2))
    model_results[model_key]={
        'trials':trials_one,'predictions':preds_one,'best':best_one
    }
    display(best_one[['model_key','feature_set','params_json','MAE','RMSE','R2']])
    print('Saved:',model_dir)
    return best_one


### 3.1 — Baseline

Uses the median training target as a simple benchmark. No optimization over the test period.

In [ ]:
MODEL_KEY = 'historical_median'
historical_median_best = run_model_experiment(MODEL_KEY, 'Historical median')

### 3.2 — Linear regression

Fits a standard linear model as a transparent predictive comparator.

In [ ]:
MODEL_KEY = 'linear_regression'
linear_regression_best = run_model_experiment(MODEL_KEY, 'Linear Regression')

### 3.3 — Ridge regression

Tunes regularization alpha on validation data for each feature set.

In [ ]:
MODEL_KEY = 'ridge'
ridge_best = run_model_experiment(MODEL_KEY, 'Ridge')

### 3.4 — Random Forest

Compares tree depth and minimum leaf sizes with a fixed random seed.

In [ ]:
MODEL_KEY = 'random_forest'
random_forest_best = run_model_experiment(MODEL_KEY, 'Random Forest')

### 3.5 — Gradient Boosting

Compares boosting learning rates and iteration counts using the same validation rows.

In [ ]:
MODEL_KEY = 'gradient_boosting'
gradient_boosting_best = run_model_experiment(MODEL_KEY, 'Gradient Boosting')

### 3.6 — Gradient Boosting with absolute-error loss (official candidate)

Uses loss aligned with MAE. The hyperparameter grid is tuned on development validation quarters only. Fits and predictions are saved alongside the other algorithms.

In [ ]:
MODEL_KEY='gb_absolute_error'
gb_absolute_error_best=run_model_experiment(MODEL_KEY,'GB absolute error')

### 3.7 — Huber regression (official candidate)

A robust linear model that downweights large residuals; fitted preprocessing is learned only on the training split.

In [ ]:
MODEL_KEY='huber'
huber_best=run_model_experiment(MODEL_KEY,'Huber')

## 4. Compare all validated models

This cell combines all independently saved experiment results, **without re-training models**. It reports every configuration and a compact best-per-model/per-feature-set table. The primary winner is the configuration with the lowest validation MAE (RMSE tie-breaker).

The comparison cannot be considered a final test result. The test partition remains reserved for Notebook 03.


In [ ]:
expected_models = ['historical_median','linear_regression','ridge',
                   'random_forest','gradient_boosting','gb_absolute_error','huber']
missing_models=[m for m in expected_models if m not in model_results]
if missing_models:
    raise RuntimeError('Run these model cells before comparing: '+', '.join(missing_models))

trials=pd.concat([model_results[m]['trials'] for m in expected_models],
                 ignore_index=True).sort_values(['MAE','RMSE','key']).reset_index(drop=True)
all_predictions=pd.concat([model_results[m]['predictions'] for m in expected_models],
                          ignore_index=True)
comparison=pd.concat([model_results[m]['best'] for m in expected_models],
                     ignore_index=True).sort_values(['MAE','RMSE','key']).reset_index(drop=True)
trials.to_csv(OUT/'validation_trials.csv',index=False)
all_predictions.to_csv(OUT/'validation_predictions.csv',index=False)
comparison.to_csv(OUT/'model_comparison_summary.csv',index=False)
display(comparison[['model_key','algorithm','feature_set','params_json','MAE','RMSE','R2']])
print('All configurations:',len(trials),' | Top validation configuration:',trials.iloc[0]['key'])


## 5. Diagnostic evaluation: why are the errors so high?

These analyses use the **already saved validation predictions** of the best validation
configuration for each algorithm on the enhanced feature set. They do not train new models,
change the primary MAE selection criterion, or inspect reserved test outcomes.

Outputs include robust error percentiles, absolute-error distributions by observed growth
range, by industry, by validation quarter, and two tail-sensitivity checks. Thresholds are
computed using *training data only*. The original, unfiltered validation results are retained.


In [ ]:
import matplotlib.pyplot as plt
DIAG_OUT=OUT/'validation_diagnostics'
DIAG_OUT.mkdir(parents=True,exist_ok=True)
enhanced_best=(trials.loc[trials.feature_set.eq('reported_plus_engineered')]
               .sort_values(['MAE','RMSE','key']).drop_duplicates('algorithm'))
assert enhanced_best.algorithm.nunique()==len(expected_models)
validation_predictions=all_predictions.loc[
    all_predictions.key.isin(enhanced_best.key)
].copy()
# Align by immutable row order within each key. Predictions are created from valid in that order.
assert all(len(validation_predictions[validation_predictions.key.eq(k)])==len(valid)
           for k in enhanced_best.key)
base_info=valid[['COMPANY_KEY','target_q','INDEX_INDUSTRY','PAT','target_growth_pct']].reset_index(drop=True)
diagnostic_rows=[]
for r in enhanced_best.itertuples():
    p=validation_predictions.loc[validation_predictions.key.eq(r.key)].reset_index(drop=True)
    assert np.allclose(p.actual.to_numpy(float),base_info.target_growth_pct.to_numpy(float))
    if not np.array_equal(p.COMPANY_KEY.astype(str).to_numpy(),
                          base_info.COMPANY_KEY.astype(str).to_numpy()):
        raise ValueError('Prediction row order/company mismatch: '+r.algorithm)
    t=base_info.copy()
    t['algorithm']=r.algorithm
    t['key']=r.key
    t['predicted']=p.predicted.to_numpy(float)
    t['signed_error']=t.predicted-t.target_growth_pct
    t['abs_error']=t.signed_error.abs()
    diagnostic_rows.append(t)
diag=pd.concat(diagnostic_rows,ignore_index=True)
diag['growth_band']=pd.cut(diag.target_growth_pct,
    bins=[-np.inf,-100,-25,0,25,100,500,np.inf],
    labels=['<-100%','-100 to -25%','-25 to 0%','0 to 25%',
            '25 to 100%','100 to 500%','>500%'])
diag['abs_growth']=diag.target_growth_pct.abs()
def trimmed_abs_error(x):
    x=np.sort(np.asarray(x,dtype=float))
    cut=int(np.floor(len(x)*.05))
    return float(x[cut:len(x)-cut].mean()) if len(x)>2*cut else np.nan
robust_metrics=diag.groupby('algorithm').agg(
    n=('abs_error','size'),MAE=('abs_error','mean'),
    median_abs_error=('abs_error','median'),
    p75_abs_error=('abs_error',lambda s:s.quantile(.75)),
    p90_abs_error=('abs_error',lambda s:s.quantile(.90)),
    p95_abs_error=('abs_error',lambda s:s.quantile(.95)),
    p99_abs_error=('abs_error',lambda s:s.quantile(.99)),
    mean_signed_error=('signed_error','mean'),
    trimmed_5pct_MAE=('abs_error',trimmed_abs_error)
).reset_index().sort_values('MAE')
robust_metrics.to_csv(DIAG_OUT/'robust_validation_metrics.csv',index=False)
display(robust_metrics)
by_growth=diag.groupby(['algorithm','growth_band'],observed=True).agg(
    n=('abs_error','size'),MAE=('abs_error','mean'),
    median_abs_error=('abs_error','median')).reset_index()
by_quarter=diag.groupby(['algorithm','target_q']).agg(
    n=('abs_error','size'),MAE=('abs_error','mean'),
    median_abs_error=('abs_error','median')).reset_index()
by_industry=diag.groupby(['algorithm','INDEX_INDUSTRY']).agg(
    n=('abs_error','size'),MAE=('abs_error','mean'),
    median_abs_error=('abs_error','median')).reset_index()
by_industry['low_support']=by_industry.n<20
by_growth.to_csv(DIAG_OUT/'error_by_growth_band.csv',index=False)
by_quarter.to_csv(DIAG_OUT/'error_by_target_quarter.csv',index=False)
by_industry.to_csv(DIAG_OUT/'error_by_industry.csv',index=False)
display(by_growth)
display(by_quarter)
display(by_industry.sort_values(['algorithm','MAE'],ascending=[True,False]).head(25))
train_abs_growth=train.target_growth_pct.abs()
growth_cap=float(train_abs_growth.quantile(.95))
positive_pat=train.PAT.abs().dropna()
positive_pat=positive_pat[positive_pat>0]
assert len(positive_pat)>0
pat_floor=float(positive_pat.quantile(.05))
sensitivity_rows=[]
for algorithm,part in diag.groupby('algorithm'):
    for label,mask in [
        ('original_all',np.ones(len(part),dtype=bool)),
        ('exclude_above_train_95pct_abs_growth',part.abs_growth<=growth_cap),
        ('exclude_below_train_5pct_abs_PAT',part.PAT.abs()>pat_floor)]:
        sub=part.loc[mask]
        sensitivity_rows.append({'algorithm':algorithm,'scenario':label,
            'n':len(sub),'fraction_kept':len(sub)/len(part),
            'MAE':float(sub.abs_error.mean()) if len(sub) else np.nan,
            'median_abs_error':float(sub.abs_error.median()) if len(sub) else np.nan,
            'RMSE':float(np.sqrt(np.mean(sub.signed_error**2))) if len(sub) else np.nan,
            'R2':float(r2_score(sub.target_growth_pct,sub.predicted)) if len(sub)>1 else np.nan})
sensitivity=pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(DIAG_OUT/'target_tail_sensitivity.csv',index=False)
display(sensitivity)
fig,ax=plt.subplots(figsize=(10,5))
for algorithm,g in by_growth.groupby('algorithm'):
    ax.plot(g.growth_band.astype(str),g.MAE,marker='o',label=algorithm)
ax.set(title='Validation MAE by target growth range (descriptive)',
       xlabel='Actual next-quarter PAT growth band',ylabel='MAE (percentage points)')
ax.tick_params(axis='x',rotation=30)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(DIAG_OUT/'validation_errors_by_growth_band.png')
plt.show()
print('Training-defined sensitivity thresholds:',{'abs_growth_95pct':growth_cap,'abs_PAT_5pct':pat_floor})
print('No filtering was applied to the primary validation comparison.')

## 6. Data sufficiency: chronological learning curves

This experiment fits selected algorithms on progressively longer histories of **training quarters**
and scores them on the *same* validation quarters. It reports training-row count, number of firms,
MAE and RMSE for each history length. It **does not** demonstrate that data volume is the sole
cause of poor performance: adding quarters also changes time regime and company composition.

Minimum starting history: **3 training target quarters**. The validation block is not used for
fitting or feature selection. We use each algorithm's already chosen configuration for the
enhanced feature set, so this is an explanatory sensitivity analysis, **not** new optimization.


In [ ]:
LEARNING_OUT=OUT/'learning_curve_diagnostics'
LEARNING_OUT.mkdir(parents=True,exist_ok=True)
train_quarters=sorted(train.target_q.astype(str).unique())
learning_rows=[]
if len(train_quarters)<4:
    print('Too few training quarters for a useful multi-point chronological learning curve.')
else:
    # Keep validation period identical; grow only the earlier training history.
    for nquarters in range(3,len(train_quarters)+1):
        history=train.loc[train.target_q.astype(str).isin(train_quarters[:nquarters])]
        for r in enhanced_best.itertuples():
            config=json.loads(r.params_json)
            cols=FEATURE_SETS['reported_plus_engineered']
            mdl=pipeline(estimator(r.algorithm,config),cols)
            mdl.fit(history[cols],history.target_growth_pct)
            p=mdl.predict(valid[cols])
            learning_rows.append({'algorithm':r.algorithm,
                'training_quarters':nquarters,'train_n':len(history),
                'train_companies':history.COMPANY_KEY.nunique(),
                'validation_n':len(valid),**metrics(valid.target_growth_pct,p)})
    learning_curve=pd.DataFrame(learning_rows)
    learning_curve.to_csv(LEARNING_OUT/'chronological_learning_curve.csv',index=False)
    display(learning_curve)
    fig,ax=plt.subplots(figsize=(9,5))
    for algorithm,g in learning_curve.groupby('algorithm'):
        ax.plot(g.training_quarters,g.MAE,marker='o',label=algorithm)
    ax.set(xlabel='Training history (number of target quarters)',
           ylabel='MAE on fixed validation quarters',
           title='Chronological learning curve: time history vs error')
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(LEARNING_OUT/'chronological_learning_curve.png')
    plt.show()
print('This diagnostic cannot separately identify sample-size effects from changing market regimes.')

## 7. Additional hyperparameter tuning: expanding-window time folds

A second controlled experiment tests whether the **original narrow tuning grids** missed
better Ridge, Random Forest or Gradient Boosting configurations. Each candidate parameter
set is evaluated using **expanding-window chronological folds inside the training partition
only** (minimum three prior target quarters). The best cross-validation configuration is
then refitted on the full train partition and scored once on the existing validation block.

Unlike choosing the lowest validation score across a bigger grid, CV tunes parameters
**without fitting or selecting on validation outcomes**. Results are saved as a *diagnostic
comparison*; the existing primary model selection and manifest are unchanged.
If you later want this expanded search to become the official selection protocol, rerun the
full model-selection workflow from a declared configuration and keep test outcomes unseen.


In [ ]:
from sklearn.model_selection import ParameterGrid
TUNING_OUT=OUT/'expanded_tuning_diagnostics'
TUNING_OUT.mkdir(parents=True,exist_ok=True)
extra_grids={
 'Ridge':list(ParameterGrid({'alpha':[0.1,1.0,10.0,100.0,1000.0]})),
 'Random Forest':list(ParameterGrid({
     'n_estimators':[150,300],'max_depth':[3,7,12],
     'min_samples_leaf':[5,20],'max_features':[0.6]})),
 'Gradient Boosting':list(ParameterGrid({
     'n_estimators':[100,250],'learning_rate':[0.02,0.08],
     'max_depth':[2,3],'min_samples_leaf':[10]}))
}
folds=[]
for cut in range(3,len(train_quarters)):
    hist=train.loc[train.target_q.astype(str).isin(train_quarters[:cut])]
    future=train.loc[train.target_q.astype(str).eq(train_quarters[cut])]
    if len(hist)>=80 and len(future)>=10:
        folds.append((hist,future,train_quarters[cut]))
print('Expanding folds:',len(folds))
if len(folds)<2:
    print('Fewer than two usable expanding folds; expanded tuning comparison is insufficient.')
else:
    cv_rows=[]
    for algorithm,grid in extra_grids.items():
        cols=FEATURE_SETS['reported_plus_engineered']
        for params in grid:
            fold_mae=[]
            for hist,future,quarter in folds:
                mdl=pipeline(estimator(algorithm,params),cols)
                mdl.fit(hist[cols],hist.target_growth_pct)
                pred=mdl.predict(future[cols])
                fold_mae.append(mean_absolute_error(future.target_growth_pct,pred))
            cv_rows.append({'algorithm':algorithm,
                'params_json':json.dumps(params,sort_keys=True),
                'cv_mean_MAE':float(np.mean(fold_mae)),
                'cv_median_MAE':float(np.median(fold_mae)),
                'cv_std_MAE':float(np.std(fold_mae)),
                'folds':len(fold_mae)})
    cv_table=pd.DataFrame(cv_rows).sort_values(
        ['algorithm','cv_mean_MAE','params_json'])
    cv_table.to_csv(TUNING_OUT/'expanding_window_cv_all_configs.csv',index=False)
    tuning_winners=cv_table.drop_duplicates('algorithm').copy()
    validation_tuning=[]
    for r in tuning_winners.itertuples():
        cols=FEATURE_SETS['reported_plus_engineered']
        mdl=pipeline(estimator(r.algorithm,json.loads(r.params_json)),cols)
        mdl.fit(train[cols],train.target_growth_pct)
        pred=mdl.predict(valid[cols])
        old=enhanced_best.loc[enhanced_best.algorithm.eq(r.algorithm)].iloc[0]
        validation_tuning.append({
            'algorithm':r.algorithm,'expanded_cv_best_params':r.params_json,
            'expanded_cv_mean_MAE':r.cv_mean_MAE,
            'new_validation_MAE':mean_absolute_error(valid.target_growth_pct,pred),
            'original_grid_validation_MAE':float(old.MAE),
            'new_minus_original_validation_MAE':float(
                mean_absolute_error(valid.target_growth_pct,pred)-old.MAE)})
    tuning_comparison=pd.DataFrame(validation_tuning)
    tuning_comparison.to_csv(TUNING_OUT/'expanded_vs_original_validation.csv',index=False)
    display(tuning_comparison)
    print('Experimental tuning check only; does not replace frozen model choices.')


## 8. Diagnostic interpretation guidance

After executing the three checks, examine these questions:

- **Target extremes?** Do the median and trimmed absolute errors look much smaller
  than MAE, and do the growth-band/tail-filter tables show concentrated large errors?
- **Limited data or time coverage?** Does validation MAE improve consistently as more
  *chronologically earlier* quarters enter training? If not, do not claim more data alone
  would solve the problem.
- **Insufficient tuning?** Do expanding-window CV-selected parameters perform better on
  the unchanged validation period? If not, a larger random parameter search is unlikely
  to be the first priority.
- **Generalization?** With only a few validation quarters, CV and validation rankings may
  be unstable. None of these checks establishes final test performance.

**Do not overwrite the original `validation_trials.csv` or choose a new primary winner
from these post-hoc diagnostics.** Document findings, revise the modeling protocol if
necessary, and rerun the predeclared workflow before Notebook 03.


## 9. New diagnostic experiment — robust losses and minimum current-PAT eligibility

**Why:** The original validation MAE was dominated by a small number of very large next-quarter PAT-growth outcomes; ordinary least squares also became unstable after adding QoQ/YoY features. We therefore run a **separate, explicitly exploratory** experiment without modifying the original validation trials, the selected primary model, the reserved test split, or the original target definition.

Two **prediction populations** are evaluated:

1. **Full population:** use the existing train and validation records.
2. **Minimum current-PAT population:** calculate the 5th percentile of **nonzero absolute current-quarter PAT using the train partition only**. Exclude records with missing PAT or abs(PAT) at or below that frozen threshold from **both fitting and evaluation**. This rule can be used at forecast time because it uses current-quarter PAT, *not future PAT growth*. It changes the population and must be reported as a restricted-coverage experiment.

Compare these candidate learning approaches, separately for each population:

- Historical median baseline (always report it).
- Conventional Ridge (same training/validation features as the robust models).
- Huber regression (reduced leverage for large residuals).
- Gradient boosting with **absolute-error** loss (target-aligned robust tree boosting).
- Ridge regression fitted on an **asinh-transformed** outcome, with predictions inverse-transformed back to original PAT-growth percentage points.

Use the already-defined **`reported_plus_engineered`** feature set for this controlled comparison. The existing seasonality experiment is separate. No new candidate is promoted into the final model-selection manifest merely because its validation metric looks favorable.

**Comparison rule:** All models in a population are compared on identical validation rows. To assess the benefit of *refitting* for the restricted population, also score full-population-trained models on that **same restricted validation cohort**; do not compare full-population and restricted-population MAEs as if they measured identical forecasting tasks. No exclusion based on *actual next-quarter growth* is used. The reserved test set remains untouched.

In [ ]:
from sklearn.linear_model import HuberRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import median_absolute_error
import warnings

ROBUST_OUT = OUT / 'robust_pat_experiments'
ROBUST_OUT.mkdir(parents=True, exist_ok=True)
ROBUST_FEATURES = FEATURE_SETS['reported_plus_engineered']
assert {'PAT','COMPANY_KEY','target_q','target_growth_pct'}.issubset(train.columns)
assert all(c in train.columns and c in valid.columns for c in ROBUST_FEATURES)

# Freeze this eligibility rule on historical TRAIN PAT only.
train_pat = pd.to_numeric(train['PAT'], errors='coerce').abs().replace([np.inf, -np.inf], np.nan)
nonzero_train_pat = train_pat.loc[train_pat.gt(0)].dropna()
assert len(nonzero_train_pat) >= 30, 'Insufficient positive absolute PAT observations.'
CURRENT_PAT_THRESHOLD = float(nonzero_train_pat.quantile(0.05))

def eligible_current_pat(frame):
    values = pd.to_numeric(frame['PAT'], errors='coerce').abs()
    return values.notna() & values.gt(CURRENT_PAT_THRESHOLD)

train_eligible = train.loc[eligible_current_pat(train)].copy()
valid_eligible = valid.loc[eligible_current_pat(valid)].copy()
if len(train_eligible) < 80 or len(valid_eligible) < 20:
    raise ValueError('Current-PAT restriction leaves too few observations; do not interpret this experiment.')
assert set(valid_eligible.index).issubset(set(valid.index))
assert set(train_eligible.index).issubset(set(train.index))
print('Train-derived minimum absolute current PAT:', CURRENT_PAT_THRESHOLD)
print('Full population:', len(train), 'train /', len(valid), 'validation')
print('Eligible population:', len(train_eligible), 'train /', len(valid_eligible), 'validation')

# This is not a hyperparameter search; use a small fixed, documented set of candidates.
def robust_estimator(name):
    if name == 'Historical median': return DummyRegressor(strategy='median')
    if name == 'Ridge': return Ridge(alpha=100.0)
    if name == 'Huber': return HuberRegressor(epsilon=1.35, alpha=10.0, max_iter=2000)
    if name == 'GB absolute error':
        return GradientBoostingRegressor(loss='absolute_error', random_state=SEED,
            n_estimators=150, learning_rate=0.05, max_depth=2, min_samples_leaf=15)
    raise ValueError(name)

ROBUST_ALGORITHMS = ['Historical median', 'Ridge', 'Huber', 'GB absolute error']
fit_models = {}
experiment_predictions = []
experiment_scores = []

# Explicitly save population-level selection/coverage; never use PAT_next to filter.
populations = {
    'full': (train, valid),
    'minimum_current_PAT': (train_eligible, valid_eligible)
}

def score_robust(y, prediction):
    y = np.asarray(y, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    if not np.isfinite(prediction).all():
        raise ValueError('Nonfinite model prediction; inspect the target transform/model.')
    absolute_error = np.abs(prediction-y)
    return {'MAE': float(np.mean(absolute_error)),
            'median_absolute_error': float(np.median(absolute_error)),
            'RMSE': float(np.sqrt(np.mean((prediction-y)**2))),
            'R2': float(r2_score(y, prediction)),
            'p95_absolute_error': float(np.quantile(absolute_error, 0.95))}

for population_name, (training_rows, evaluation_rows) in populations.items():
    for algorithm in ROBUST_ALGORITHMS:
        fitted = pipeline(robust_estimator(algorithm), ROBUST_FEATURES)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            fitted.fit(training_rows[ROBUST_FEATURES], training_rows.target_growth_pct)
        pred = fitted.predict(evaluation_rows[ROBUST_FEATURES])
        name = population_name + '|' + algorithm
        fit_models[name] = fitted
        scores = score_robust(evaluation_rows.target_growth_pct, pred)
        experiment_scores.append({'population':population_name, 'algorithm':algorithm,
            'train_n':len(training_rows), 'validation_n':len(evaluation_rows),
            'train_companies':training_rows.COMPANY_KEY.nunique(),
            'validation_companies':evaluation_rows.COMPANY_KEY.nunique(),
            'training_warning_count':len(caught), **scores})
        experiment_predictions.append(pd.DataFrame({
            'population': population_name, 'algorithm': algorithm,
            'COMPANY_KEY': evaluation_rows.COMPANY_KEY.to_numpy(),
            'target_q': evaluation_rows.target_q.astype(str).to_numpy(),
            'actual': evaluation_rows.target_growth_pct.to_numpy(),
            'predicted': pred}))
        if caught:
            print(name, 'fit warnings:', sorted(set(type(w.message).__name__ for w in caught)))

robust_scores = pd.DataFrame(experiment_scores).sort_values(['population','MAE','algorithm'])
robust_predictions = pd.concat(experiment_predictions, ignore_index=True)
robust_scores.to_csv(ROBUST_OUT/'robust_validation_metrics.csv', index=False)
robust_predictions.to_csv(ROBUST_OUT/'robust_validation_predictions.csv', index=False)
display(robust_scores)

# Exactly matched eligible validation cohort for both fitting choices.
matched_scores = []
for algorithm in ROBUST_ALGORITHMS:
    y = valid_eligible.target_growth_pct.to_numpy(float)
    for source_population in ['full','minimum_current_PAT']:
        fitted = fit_models[source_population+'|'+algorithm]
        pred = fitted.predict(valid_eligible[ROBUST_FEATURES])
        matched_scores.append({'algorithm':algorithm, 'trained_on':source_population,
            'evaluated_on':'minimum_current_PAT', 'validation_n':len(valid_eligible),
            **score_robust(y,pred)})
matched_comparison=pd.DataFrame(matched_scores).sort_values(['algorithm','trained_on'])
matched_comparison.to_csv(ROBUST_OUT/'matched_eligible_validation_comparison.csv',index=False)
display(matched_comparison)

coverage = pd.DataFrame([
    {'population':'full', 'train_rows':len(train),'validation_rows':len(valid),
     'validation_coverage':1.0},
    {'population':'minimum_current_PAT','train_rows':len(train_eligible),
     'validation_rows':len(valid_eligible),
     'validation_coverage':len(valid_eligible)/len(valid)}
])
coverage.to_csv(ROBUST_OUT/'population_coverage.csv',index=False)
print('Threshold is based on current-quarter PAT; do NOT use next-quarter target to restrict cases.')

## 10. Interpretation and diagnostic exports

- **`robust_validation_metrics.csv`**: main and robust regression metrics for the full and train-PAT-restricted populations. Within each population, algorithms use identical validation observations.
- **`matched_eligible_validation_comparison.csv`**: full-trained versus eligible-trained models scored on exactly the **same eligible validation rows** (the fair way to isolate refitting/eligibility effects).
- **`population_coverage.csv`**: proportion of companies/observations retained by the minimum current-PAT criterion.
- **`robust_validation_predictions.csv`**: individual observations for error inspection. Treat these as development-stage results.

A lower restricted-population MAE by itself does **not** prove better modeling; it may simply reflect a less challenging cohort. Huber convergence warnings and the `asinh` inverse-transform should be reviewed for nonfinite or extreme forecasts. If models truly improve on the fixed validation protocol, revise the frozen selection protocol **before** opening the test set and re-execute model selection. This diagnostic section intentionally does not modify Notebook 03's frozen primary model.

## 11. Freeze primary model, RQ3 candidate keys and RQ4 Ridge comparator

In [ ]:
# FINAL MODEL SELECTION: validation only. Do not change after looking at test performance.
# All model families (including robust models) competed over the same validation rows.
primary_winner=trials.sort_values(['MAE','RMSE','key']).iloc[0]
comparison_fs='reported_plus_engineered'
by_algorithm=(trials.loc[trials.feature_set.eq(comparison_fs)]
    .sort_values(['MAE','RMSE','key']).drop_duplicates('algorithm'))
assert {'Historical median','GB absolute error','Huber'}.issubset(set(by_algorithm.algorithm))

# RQ4 requires an identical algorithm across feature sets.
# Predeclare GB absolute error (a nonconstant robust family) for all comparisons.
rq4_algorithm='GB absolute error'
by_features=(trials.loc[trials.algorithm.eq(rq4_algorithm)]
    .sort_values(['MAE','RMSE','key']).drop_duplicates('feature_set'))
required_sets={'reported_only','reported_plus_engineered','reported_plus_engineered_plus_seasonal'}
assert required_sets.issubset(set(by_features.feature_set))
rq4_keys=by_features.set_index('feature_set').key.to_dict()
selection={
  'seed':SEED,'source_sha256':source_hash,'feature_sets':FEATURE_SETS,
  'criterion':'Lowest validation MAE, then RMSE; all test outcomes untouched',
  'predeclared_rq4_algorithm':rq4_algorithm,
  'primary':{'key':str(primary_winner.key),'algorithm':str(primary_winner.algorithm),
             'feature_set':str(primary_winner.feature_set)},
  'rq3_keys':by_algorithm.set_index('algorithm').key.to_dict(),
  'rq4_keys':{x:rq4_keys[x] for x in ['reported_only','reported_plus_engineered']},
  'seasonality_keys':{x:rq4_keys[x] for x in ['reported_plus_engineered','reported_plus_engineered_plus_seasonal']},
  'qoq_feature_config':{'source_columns':QOQ_SOURCE_COLUMNS,'suffixes':QOQ_SUFFIXES,
      'generation_rule':'exact prior quarter, no missing-quarter carryover'},
  'seasonality_feature_config':{'source_columns':SEASONAL_SOURCE_COLUMNS,
      'suffixes':SEASONAL_SUFFIXES,'calendar_columns':SEASONAL_CALENDAR,
      'generation_rule':'exact same calendar quarter previous year'},
  'training_feature_eligibility_file':'training_feature_eligibility.csv',
  'train_target_quarters':sorted(q['train']),
  'validation_target_quarters':sorted(q['validation']),
  'test_target_quarters':sorted(q['test']),
  'minimum_current_pat_threshold':float(CURRENT_PAT_THRESHOLD),
  'secondary_population_policy':'abs(current PAT) > training nonzero abs(PAT) 5th percentile',
  'limitations':['Only two validation target quarters','Extreme PAT-growth denominators',
    'Development-selected comparator families may still have model uncertainty']
}
(OUT/'selection_manifest.json').write_text(json.dumps(selection,indent=2))
by_features.to_csv(OUT/'seasonality_validation_comparison.csv',index=False)
print('Frozen PRIMARY:',selection['primary'])
display(by_algorithm[['algorithm','feature_set','MAE','RMSE','R2']])
display(by_features[['feature_set','algorithm','MAE','RMSE','R2']])
print('Frozen manifest:',OUT/'selection_manifest.json')


## 12. Financial Services-only exploratory banking comparison

In [ ]:
# Standalone exploratory sector comparison: not used to choose the overall winner.
sector='financial services'
btr=train[train.INDEX_INDUSTRY.str.strip().str.casefold().eq(sector)].copy()
bva=valid[valid.INDEX_INDUSTRY.str.strip().str.casefold().eq(sector)].copy()
core_b,core_detail=eligibility(btr,CORE,60,15)
bank_b,bank_detail=eligibility(btr,BANK,40,10)
bank_qoq,bank_qoq_detail=eligibility(btr,QOQ_BANK_ENGINEERED,40,10)
pd.concat([core_detail.assign(group='core'),bank_detail.assign(group='banking'),
           bank_qoq_detail.assign(group='banking_qoq')]).to_csv(
    OUT/'banking_feature_eligibility.csv',index=False)
bank_selection={'status':'skipped','reason':'Insufficient eligible training/validation rows or features'}
if len(btr)>=80 and btr.COMPANY_KEY.nunique()>=20 and len(bva)>=20 and core_b and bank_b:
 bank_sets={'bank_core_only':core_b+CATS,'bank_core_plus_banking':core_b+bank_b+bank_qoq+CATS}
 bank_scores=[]
 for group,cols in bank_sets.items():
  for param in GRID['Ridge']:
   m=pipeline(estimator('Ridge',param),cols);m.fit(btr[cols],btr.target_growth_pct)
   score=metrics(bva.target_growth_pct,m.predict(bva[cols]))
   bank_scores.append({'feature_set':group,'params_json':json.dumps(param,sort_keys=True),**score})
 scores=pd.DataFrame(bank_scores).sort_values(['MAE','RMSE','feature_set'])
 scores.to_csv(OUT/'banking_validation_trials.csv',index=False)
 winners=scores.drop_duplicates('feature_set')
 bank_selection={'status':'selected','feature_sets':bank_sets,'algorithm':'Ridge',
  'keys':{r.feature_set:f'{r.feature_set}|Ridge|{r.params_json}' for r in winners.itertuples()},
  'train_rows':int(len(btr)),'validation_rows':int(len(bva))}
 display(winners)
selection['banking_comparison']=bank_selection
(OUT/'selection_manifest.json').write_text(json.dumps(selection,indent=2))
print('Frozen manifests written:',OUT/'selection_manifest.json')

## 13. Limitations

Only a few quarters are available. Validate filing-availability dates and source-tag semantics. Extreme next-quarter PAT growth may dominate error metrics. This notebook **does not** compute final test-set scores. No results are claimed until the notebook runs successfully.

## Final handoff to Notebook 03

**Stop model development here.** This notebook exports the final validation-frozen `model_outputs/selection_manifest.json`. Notebook 03 must use exactly the saved model keys, feature lists, PAT threshold and source hash. Never inspect or tune against the reserved test target until Notebook 03. The minimum-PAT comparison is secondary; the full population remains the primary estimand.